# Predicting Student Dropout — Results Overview

This notebook is a **guided tour of the results**. It does not train any models: it reads the outputs that the pipeline in `src/` writes to `reports/`, `data/processed/` and `models/`, so what you see here always matches the code.

**To reproduce everything (from the project folder):**

```bash
pip install -r requirements.txt
python run_all.py        # runs src/01 … src/09 in order (about 10 minutes)
```

Then open this notebook and choose **Kernel → Restart & Run All**.

| Section | Source script |
| --- | --- |
| 1. The data | `01_data_quality_eda.py` |
| 2. Selected features | `02_eda_feature_engineering.py` |
| 3. Model comparison | `03`–`07` (five supervised models) |
| 4. Explainability & fairness | `09_bias_fairness_audit.py` |
| 5. Student segments | `08_unsupervised_analysis.py` |

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown

# Find the project folder whether this notebook is run from notebooks/ or the project root
ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent

REPORTS = ROOT / "reports"
FIGURES = REPORTS / "figures"

def show_image(path, width=900):
    """Show a saved chart, or explain how to create it if it is missing."""
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        display(Markdown(f"⚠️ `{path.relative_to(ROOT)}` not found — run `python run_all.py` first."))

def read_csv(path):
    if path.exists():
        return pd.read_csv(path)
    display(Markdown(f"⚠️ `{path.relative_to(ROOT)}` not found — run `python run_all.py` first."))
    return None

pd.set_option("display.precision", 3)
print("Project folder:", ROOT)

## 1. The data

4,424 students, 36 input variables and one outcome (`target`: Graduate, Dropout or Enrolled). The task is framed as **Dropout vs not Dropout**.

In [ ]:
df = pd.read_csv(ROOT / "data" / "raw" / "students_dropout_academic_success.csv")
print(f"Rows: {len(df):,}   Columns: {df.shape[1]}   Missing values: {df.isna().sum().sum()}   Duplicates: {df.duplicated().sum()}")

outcome = (df["target"].value_counts()
             .rename("Students").to_frame()
             .assign(Share=lambda t: (t["Students"] / t["Students"].sum() * 100).round(1).astype(str) + "%"))
outcome

The charts below come from the data-quality step: completeness of every column, and the outcome split.

In [ ]:
show_image(FIGURES / "section2_eda" / "Fig2_1_data_quality_overview.png")
show_image(FIGURES / "section2_eda" / "Fig2_2_target_distribution.png")

## 2. Selected features

Step 3 engineered 22 domain features (approval rates, engagement, financial hardship, etc.) and ranked all features with three selection methods. The 25 features below are the Random Forest importance ranking produced by `02_eda_feature_engineering.py`, most important first.

The model scripts (03–07) use a **fixed** list of 25 features chosen during the original analysis, so their results do not change if the selection is re-run. The cell below shows how that fixed list compares with the current ranking.

In [ ]:
feature_file = ROOT / "data" / "processed" / "feature_list.txt"
if feature_file.exists():
    features = [line.strip() for line in feature_file.read_text(encoding="utf-8").splitlines() if line.strip()]
    fixed_list = [
        'approval_rate_2nd', 'overall_approval_rate', 'Curricular units 2nd sem (approved)',
        'approval_rate_1st', 'total_approved', 'Curricular units 2nd sem (grade)',
        'failed_majority_2nd', 'avg_grade_1st_2nd', 'Tuition fees up to date',
        'Curricular units 1st sem (grade)', 'financial_hardship', 'Admission grade',
        'admission_grade_zscore', 'Previous qualification (grade)', 'Age at enrollment',
        'grade_decline', 'parental_occupation_avg', 'Course',
        'Curricular units 2nd sem (evaluations)', "Father's occupation",
        'Curricular units 1st sem (approved)', 'parental_education_avg', 'GDP',
        'engagement_2nd', 'engagement_1st']   # as used in src/03-07
    ranking = pd.DataFrame({"Rank": range(1, len(features) + 1), "Feature": features})
    ranking["In model scripts' fixed list"] = ranking["Feature"].isin(fixed_list).map({True: "Yes", False: "No"})
    display(ranking.set_index("Rank"))
    shared = len(set(features) & set(fixed_list))
    only_fixed = sorted(set(fixed_list) - set(features))
    only_ranked = sorted(set(features) - set(fixed_list))
    note = f"The fixed list shares **{shared} of 25** features with this ranking."
    if only_fixed:
        note += f" Only in the fixed list: {', '.join(only_fixed)}. Only in this ranking: {', '.join(only_ranked)}."
    display(Markdown(note))
else:
    display(Markdown("⚠️ `data/processed/feature_list.txt` not found — run `python run_all.py` first."))

## 3. Model comparison

Five supervised models were trained on the same 64 / 16 / 20 stratified train / validation / test split (`random_state = 42`).

* **Cut-off:** a student is flagged as at-risk when the predicted dropout probability is at or above the cut-off. Logistic Regression uses 0.50. For the other four models the cut-off was **chosen on the validation set** (the highest cut-off whose validation recall reaches 80%) and then applied once to the test set — the test set was never used to choose it.
* **Targets set in Step 1:** recall ≥ 80%, precision ≥ 70%, AUC-ROC ≥ 0.85 (Dropout class).

All numbers below are **test-set** results read from `reports/model_metrics.csv`.

In [ ]:
metrics = read_csv(REPORTS / "model_metrics.csv")
TARGETS = {"recall": 0.80, "precision": 0.70, "auc_roc": 0.85}

if metrics is not None:
    table = metrics.copy()
    table["threshold"] = table["threshold"].fillna(0.50)
    table["All targets met"] = ((table["recall"] >= TARGETS["recall"]) &
                                (table["precision"] >= TARGETS["precision"]) &
                                (table["auc_roc"] >= TARGETS["auc_roc"])).map({True: "Yes", False: "No"})
    table = table.rename(columns={"model": "Model", "threshold": "Cut-off", "accuracy": "Accuracy",
                                  "precision": "Precision", "recall": "Recall", "f1": "F1",
                                  "auc_roc": "AUC-ROC"})
    display(table[["Model", "Cut-off", "Precision", "Recall", "F1", "AUC-ROC", "Accuracy", "All targets met"]]
            .set_index("Model"))

In [ ]:
if metrics is not None:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), sharey=False)
    for ax, (col, label) in zip(axes, [("recall", "Recall"), ("precision", "Precision"), ("auc_roc", "AUC-ROC")]):
        values = metrics.set_index("model")[col]
        colors = ["#2e7d32" if v >= TARGETS[col] else "#c62828" for v in values]
        ax.barh(values.index, values.values, color=colors)
        ax.axvline(TARGETS[col], color="black", linestyle="--", linewidth=1)
        ax.set_xlim(0.5 if col != "auc_roc" else 0.8, 1.0)
        ax.set_title(f"{label} (target ≥ {TARGETS[col]:.2f}, dashed)")
        ax.invert_yaxis()
        for i, v in enumerate(values.values):
            ax.text(v + 0.003, i, f"{v:.3f}", va="center", fontsize=9)
        if ax is not axes[0]:
            ax.set_yticklabels([])
    fig.suptitle("Test-set performance by model (green = meets target, red = below)", fontweight="bold")
    plt.tight_layout()
    plt.show()

In [ ]:
if metrics is not None:
    best_auc = metrics.loc[metrics["auc_roc"].idxmax()]
    missed = metrics[metrics["recall"] < TARGETS["recall"]]
    lines = [f"- Highest AUC-ROC: **{best_auc['model']}** ({best_auc['auc_roc']:.3f}).",
             f"- Models meeting all three targets: **{(table['All targets met'] == 'Yes').sum()} of {len(table)}**."]
    if len(missed):
        lines.append("- Recall below the 80% target: " +
                     ", ".join(f"**{r.model}** ({r.recall:.1%})" for r in missed.itertuples()) + ".")
    display(Markdown("\n".join(lines)))

**Why Logistic Regression is the selected model:** it meets all three targets with the default cut-off, and its coefficients explain each prediction directly — important when advisors must justify an intervention to a student. The tree ensembles reach slightly higher AUC but are harder to explain.

## 4. Explainability and fairness

The Step 5 audit (`09_bias_fairness_audit.py`) trains its own Logistic Regression on all 40 features (36 original + 4 engineered) and an XGBoost comparator, then explains and audits them.

* **(a) SHAP summary:** which features push predictions towards or away from Dropout.
* **(b, c) Partial dependence + ICE:** how predicted dropout risk changes with the 2nd-semester approval rate and with age at enrolment.

In [ ]:
show_image(FIGURES / "step5_explainability.png", width=1000)

Fairness results for the baseline Logistic Regression (cut-off 0.50). `flag` is the share of each group the model flags as at-risk, `base` the actual dropout rate, `TPR` the share of actual dropouts that are caught, `FPR` the share of non-dropouts wrongly flagged.

In [ ]:
results_log = REPORTS / "step5_results.txt"
if results_log.exists():
    text = results_log.read_text(encoding="utf-8").splitlines()
    start = next(i for i, l in enumerate(text) if "BASELINE LR" in l)
    end = next(i for i, l in enumerate(text) if "95% bootstrap" in l)
    print("\n".join(text[start:end]))
    print(f"\nFull audit log (mitigations M1-M3, bootstrap intervals): {results_log.relative_to(ROOT)}")
else:
    display(Markdown("⚠️ `reports/step5_results.txt` not found — run `python run_all.py` first."))

## 5. Student segments (unsupervised)

K-Means groups students without using the outcome. Silhouette scores favour 2 clusters; 3 are used because they give practical **low / medium / high** risk tiers for planning interventions. Risk tiers are ranked by each cluster's actual dropout rate.

In [ ]:
profiles = read_csv(REPORTS / "unsupervised_cluster_profiles.csv")
if profiles is not None:
    display(profiles.sort_values("Dropout Rate").set_index("Cluster").round(1))
show_image(FIGURES / "unsupervised" / "unsupervised_pca_analysis.png", width=900)

## 6. Limitations to keep in mind

* **Single institution.** All data comes from one higher-education institution, so results may not transfer elsewhere.
* **Timing of features.** The strongest predictors are 2nd-semester results, which are only available mid-year. The Step 5 ablation shows how much performance remains using enrolment-only or 1st-semester data.
* **Two Logistic Regression configurations.** Section 3 uses the 25 selected features (Step 4); Section 4 uses all 40 features (Step 5 audit). Their numbers differ slightly by design.
* **Validation set double duty (neural network).** The network also uses the validation set for early stopping, so its cut-off choice is slightly less independent than for the other models.
* **Hardware differences.** With fixed seeds, results repeat exactly on the same computer; the neural network may differ in the third decimal place on other hardware or library versions.